# Fermionic Gaussian preparation

Original four-mode orbital factorization, eight-mode one-electron ring, two-mode pairing Hamiltonian, Majorana canonical planes, Bogoliubov relation, and full-state validation across all 25 code stages.

The ring uses freshly computed adjacent orbital rotations from the native Fermi–Hubbard example. The two-mode case uses Hermitian iA eigenvectors to reconstruct the real canonical planes, then solves the two quasiparticle annihilator equations and synthesizes a bounded parity-preserving preparation. Direct Fock-space diagonalization is an independent oracle only. The final matrix check verifies the canonical unitary relation; it does not reproduce the OpenFermion double-Givens or particle-hole factor order. This example makes no general multi-mode pairing claim.

## Use the existing local environment

In [ ]:
from pathlib import Path

import torch
from IPython.display import Image, display


## Import the native tools

In [ ]:
from examples.synthesis.algorithms.quantum_state_preparation.fermionic_gaussian.overview.model import (
    canonicalize,
    pairing_inputs,
    periodic_hamiltonian,
    vacuum,
)
from examples.synthesis.algorithms.quantum_state_preparation.fermionic_gaussian.overview.plots import (
    plot_slater,
)
from examples.synthesis.algorithms.quantum_state_preparation.fermionic_gaussian.overview.reference import (
    annihilation,
    fock_hamiltonian,
)
from examples.synthesis.algorithms.quantum_state_preparation.fermionic_gaussian.overview.workflow import (
    demo,
    run_pairing,
    run_slater,
)

folder = Path("artifacts/development/fermionic-gaussian")


## Construct the source four-mode example

In [ ]:
print("Four-mode onsite and hopping example")
energies, occupied, error = demo()


## Read the occupied eigenmodes

In [ ]:
print("Orbital energies:", energies.tolist())
print("Occupied columns:", occupied)


## Factor the occupied block

In [ ]:
print("Use adjacent rotations and occupied-pivot signs to reconstruct the block.")


## Read the factorization residual

In [ ]:
print("Occupied-block reconstruction residual:", error)


## Verify the inverse rotations

In [ ]:
assert error < 1e-13


## Check the one-electron sector

In [ ]:
assert occupied.shape == (4, 2) and len(energies) == 4


## Retain four sites and eight modes

In [ ]:
sites, modes = 4, 8
print("Sites and modes:", sites, modes)


## Describe the mode convention

In [ ]:
print("Even mode indices carry spin-up; odd indices carry spin-down.")


## Compose the native Slater preparation

In [ ]:
print("One occupied orbital with adjacent Givens rotations.")


## Rebuild the periodic one-body model

In [ ]:
ring = periodic_hamiltonian()
print(ring)


## Check the original width

In [ ]:
assert ring.shape == (modes, modes)


## Synthesize the original Slater state

In [ ]:
slater = run_slater()
print("Slater resources:", slater.program.report["resources"])


## Read one-body energies

In [ ]:
print("Ring energies:", slater.values.tolist())


## Inspect the native amplitudes

In [ ]:
print("Native orbital amplitudes:", slater.amplitudes)


## Plot the actual state against the exact orbital

In [ ]:
display(Image(filename=str(plot_slater(slater, folder / "orbital.png"))))


## Check the one-electron fidelity

In [ ]:
print("Slater fidelity:", slater.fidelity)
assert abs(slater.fidelity - 1.0) < 1e-12


## Set up the paired Gaussian vacuum

In [ ]:
print("Solve the common kernel of the two quasiparticle annihilators.")


## Use the original hopping and pairing matrices

In [ ]:
h, delta = pairing_inputs()
print(h, delta)


## Build the independent Fock Hamiltonian

In [ ]:
hamiltonian = fock_hamiltonian(h, delta, annihilation(2))
print(torch.linalg.eigvalsh(hamiltonian))


## Synthesize the two-mode paired case

In [ ]:
paired = run_pairing()
print("Paired resources:", paired.program.report["resources"])


## Check the full paired state

In [ ]:
print("Paired fidelity:", paired.fidelity)
print("Fock amplitudes:", paired.amplitudes)
assert abs(paired.fidelity - 1.0) < 1e-12


## Inspect the original canonical decomposition

In [ ]:
decomposition = canonicalize(h, delta)
print("Canonical positive energies:", decomposition.energies)
print(decomposition.blocks)


## Verify the reconstructed transformation and vacuum

In [ ]:
w = decomposition.bogoliubov
identity = torch.eye(4, dtype=torch.complex128)
torch.testing.assert_close(w[2:] @ w.mH, identity[2:], rtol=0, atol=1e-13)
torch.testing.assert_close(
    decomposition.transform.T @ decomposition.blocks @ decomposition.transform,
    decomposition.generator,
    rtol=0,
    atol=1e-13,
)
_, residual = vacuum(decomposition)
print("Vacuum annihilator residual:", residual)
assert residual < 1e-12
